In [3]:
#@title 🔧 Установка (запусти один раз)
!pip install -q faster-whisper
!pip install -q gradio  # для удобного интерфейса

print("✅ Установка завершена!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 103.7 MB/s eta 0:00:00
✅ Установка завершена!


In [4]:
#@title 🚀 Загрузка модели Whisper Large-V3-Turbo
from faster_whisper import WhisperModel
import torch

# Проверяем GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

print(f"🖥️ Устройство: {device.upper()}")
print("⏳ Загружаю модель large-v3-turbo (это займёт 1-2 минуты)...")

model = WhisperModel(
    "large-v3-turbo",
    device=device,
    compute_type=compute_type
)

print("✅ Модель загружена и готова к работе!")

🖥️ Устройство: CUDA
⏳ Загружаю модель large-v3-turbo (это займёт 1-2 минуты)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Модель загружена и готова к работе!


In [5]:
#@title 📝 Функция транскрибации

def transcribe_audio(audio_path, language=None, task="transcribe"):
    """
    Транскрибирует аудио/видео файл.

    Args:
        audio_path: путь к файлу
        language: язык ('ru', 'en', 'uk' и т.д.) или None для автоопределения
        task: 'transcribe' (транскрибация) или 'translate' (перевод на английский)
    """
    print(f"🎙️ Обрабатываю: {audio_path}")
    print("⏳ Это может занять несколько минут...")

    segments, info = model.transcribe(
        audio_path,
        language=language,
        task=task,
        beam_size=5,
        vad_filter=True,  # убирает тишину
        vad_parameters=dict(min_silence_duration_ms=500)
    )

    print(f"📊 Обнаружен язык: {info.language} (вероятность: {info.language_probability:.1%})")
    print(f"⏱️ Длительность: {info.duration:.1f} сек\n")

    # Собираем результат
    full_text = []
    timestamps = []

    for segment in segments:
        full_text.append(segment.text.strip())
        timestamps.append({
            "start": segment.start,
            "end": segment.end,
            "text": segment.text.strip()
        })
        # Показываем прогресс
        print(f"[{segment.start:.1f}s → {segment.end:.1f}s] {segment.text.strip()}")

    return "\n".join(full_text), timestamps

print("✅ Функция готова!")

✅ Функция готова!


In [6]:
#@title 📤 Загрузи файл и получи транскрипцию
from google.colab import files
import os

# Загрузка файла
print("📤 Выбери аудио или видео файл (mp3, wav, mp4, m4a, webm и др.):")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    print(f"\n✅ Загружен: {filename}")

    #@markdown ---
    #@markdown ### Настройки:
    language = "en" #@param ["ru", "en", "uk", "auto"] {allow-input: true}
    task = "transcribe" #@param ["transcribe", "translate"]

    # Если auto - передаём None для автоопределения
    lang = None if language == "auto" else language

    # Транскрибация
    text, timestamps = transcribe_audio(filename, language=lang, task=task)

    print("\n" + "="*50)
    print("📄 ПОЛНЫЙ ТЕКСТ:")
    print("="*50)
    print(text)

📤 Выбери аудио или видео файл (mp3, wav, mp4, m4a, webm и др.):


Saving Без названия.mp3 to Без названия (2).mp3

✅ Загружен: Без названия (2).mp3
🎙️ Обрабатываю: Без названия (2).mp3
⏳ Это может занять несколько минут...
📊 Обнаружен язык: en (вероятность: 100.0%)
⏱️ Длительность: 231.6 сек

[0.0s → 7.9s] Welcome to Harvey. Over 74,000 lawyers at 700 organizations in 58 countries rely on Harvey
[7.9s → 14.2s] to accelerate their expertise across legal practice areas. Users save up to 25 hours each month,
[14.6s → 20.8s] time that can be reclaimed for high-value work. Access Harvey anywhere, on desktop, mobile,
[21.1s → 27.2s] or the Microsoft suite. Built to enterprise-grade standards, all data is encrypted, controlled,
[27.2s → 32.7s] and never used to train models. Assistant is your intelligent legal co-worker,
[33.5s → 39.7s] able to think, research, and draft like your best associate. Harvey pulls context from trusted
[39.7s → 48.1s] sources like iManage, Vault, LexisNexis, Edgar, and over 100 legal datasets to deliver cross-jurisdictional
[48.1